In [1]:
import time
from pathlib import Path

import pandas as pd
from nba_api.stats.endpoints import leaguedashteamstats

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

def get_team_stats(season, season_type):
    for attempt in range(3):
        try:
            time.sleep(2)

            df = leaguedashteamstats.LeagueDashTeamStats(
                season=season,
                season_type_all_star=season_type,
                per_mode_detailed="Totals",
                league_id_nullable="00",
                timeout=60,
            ).get_data_frames()[0]

            expected_teams = (
                16 if season_type == "Playoffs"
                else 29 if int(season[:4]) < 2004
                else 30
            )

            if len(df) != expected_teams:
                raise ValueError(
                    f"{season} {season_type}: expected "
                    f"{expected_teams} teams, received {len(df)}"
                )

            if df["TEAM_ID"].duplicated().any():
                raise ValueError("Duplicate team IDs returned")

            return df

        except Exception:
            if attempt == 2:
                raise

            print(f"Retrying {season} {season_type}...")
            time.sleep(5 * (attempt + 1))

In [2]:
season_frames = []

for year in range(2000, 2025):
    season = f"{year}-{str(year + 1)[-2:]}"
    print(f"Collecting {season}...")

    regular = get_team_stats(season, "Regular Season")
    playoffs = get_team_stats(season, "Playoffs")

    df = regular[
        ["TEAM_ID", "TEAM_NAME", "GP", "W", "L"]
    ].copy()

    df = df.rename(columns={
        "TEAM_ID": "team_id",
        "TEAM_NAME": "team_name",
        "GP": "games",
        "W": "wins",
        "L": "losses",
    })

    df["season"] = season
    df["season_start"] = year
    df["win_pct"] = df["wins"] / df["games"]
    df["made_playoffs"] = (
        df["team_id"].isin(playoffs["TEAM_ID"]).astype(int)
    )

    season_frames.append(df)

team_seasons = (
    pd.concat(season_frames, ignore_index=True)
    .sort_values(["team_id", "season_start"])
    .reset_index(drop=True)
)

assert not team_seasons.duplicated(
    ["team_id", "season_start"]
).any()

assert (
    team_seasons["wins"] + team_seasons["losses"]
    == team_seasons["games"]
).all()

assert team_seasons["win_pct"].between(0, 1).all()

output_path = DATA_DIR / "team_seasons.csv"
team_seasons.to_csv(output_path, index=False)

print(f"Saved {len(team_seasons)} rows to {output_path.resolve()}")
display(team_seasons.head())

Saved 746 rows to /Users/rajveer/Eclipse Analytics - Group 1 /data/team_seasons.csv


,team_id,team_name,games,wins,losses,season,season_start,win_pct,made_playoffs
0,1610612737,Atlanta Hawks,82,25,57,2000-01,2000,0.304878,0
1,1610612737,Atlanta Hawks,82,33,49,2001-02,2001,0.402439,0
2,1610612737,Atlanta Hawks,82,35,47,2002-03,2002,0.426829,0
3,1610612737,Atlanta Hawks,82,28,54,2003-04,2003,0.341463,0
4,1610612737,Atlanta Hawks,82,13,69,2004-05,2004,0.158537,0


In [3]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("data")
team_seasons = pd.read_csv(DATA_DIR / "team_seasons.csv")

display(team_seasons.head())

,team_id,team_name,games,wins,losses,season,season_start,win_pct,made_playoffs
0,1610612737,Atlanta Hawks,82,25,57,2000-01,2000,0.304878,0
1,1610612737,Atlanta Hawks,82,33,49,2001-02,2001,0.402439,0
2,1610612737,Atlanta Hawks,82,35,47,2002-03,2002,0.426829,0
3,1610612737,Atlanta Hawks,82,28,54,2003-04,2003,0.341463,0
4,1610612737,Atlanta Hawks,82,13,69,2004-05,2004,0.158537,0


In [4]:
LOSING_CUTOFF = 0.350
COMPARISON_MIN = 0.400
COMPARISON_MAX = 0.550
FOLLOWUP_YEARS = 5

records = []

for team_id, team in team_seasons.groupby("team_id"):
    team = team.sort_values("season_start").reset_index(drop=True)
    last_followup_end = None

    for i in range(1, len(team) - FOLLOWUP_YEARS):
        previous = team.iloc[i - 1]
        current = team.iloc[i]
        anchor_year = int(current["season_start"])

        if (
            last_followup_end is not None
            and anchor_year <= last_followup_end
        ):
            continue

        window = team.iloc[i - 1:i + FOLLOWUP_YEARS + 1]
        expected_years = list(
            range(anchor_year - 1, anchor_year + FOLLOWUP_YEARS + 1)
        )

        if window["season_start"].tolist() != expected_years:
            continue

        qualifying = team.iloc[i - 1:i + 1]
        qualifying_pct = qualifying["win_pct"]

        if (qualifying_pct < LOSING_CUTOFF).all():
            group = "sustained_losing"
        elif qualifying_pct.between(
            COMPARISON_MIN, COMPARISON_MAX
        ).all():
            group = "comparison"
        else:
            continue

        future = team.iloc[i + 1:i + FOLLOWUP_YEARS + 1]
        future_avg = future["win_pct"].mean()

        record = {
            "team_id": team_id,
            "team_name": current["team_name"],
            "qualifying_first_season": previous["season"],
            "qualifying_second_season": current["season"],
            "anchor_year": anchor_year,
            "group": group,
            "sustained_losing": int(group == "sustained_losing"),
            "previous_win_pct": previous["win_pct"],
            "anchor_win_pct": current["win_pct"],
            "qualifying_avg_win_pct": qualifying_pct.mean(),
            "followup_first_season": future.iloc[0]["season"],
            "followup_last_season": future.iloc[-1]["season"],
            "future_avg_win_pct": future_avg,
            "win_pct_change": future_avg - current["win_pct"],
            "future_playoff_appearances": int(
                future["made_playoffs"].sum()
            ),
            "made_playoffs_within_5": int(
                future["made_playoffs"].any()
            ),
        }

        for year_number, (_, future_season) in enumerate(
            future.iterrows(), start=1
        ):
            record[f"year_{year_number}_win_pct"] = (
                future_season["win_pct"]
            )
            record[f"year_{year_number}_made_playoffs"] = int(
                future_season["made_playoffs"]
            )

        records.append(record)
        last_followup_end = anchor_year + FOLLOWUP_YEARS

analysis = pd.DataFrame(records)

if analysis.empty:
    raise ValueError("No qualifying examples found. Check the input data.")

analysis.to_csv(DATA_DIR / "analysis.csv", index=False)

print(f"Saved {len(analysis)} examples to analysis.csv")
display(analysis.head())

Saved 61 examples to analysis.csv


,team_id,team_name,qualifying_first_season,qualifying_second_season,anchor_year,group,sustained_losing,previous_win_pct,anchor_win_pct,qualifying_avg_win_pct,...,year_1_win_pct,year_1_made_playoffs,year_2_win_pct,year_2_made_playoffs,year_3_win_pct,year_3_made_playoffs,year_4_win_pct,year_4_made_playoffs,year_5_win_pct,year_5_made_playoffs
0,1610612737,Atlanta Hawks,2001-02,2002-03,2002,comparison,0,0.402439,0.426829,0.414634,...,0.341463,0,0.158537,0,0.317073,0,0.365854,0,0.451220,1
1,1610612737,Atlanta Hawks,2012-13,2013-14,2013,comparison,0,0.536585,0.463415,0.500000,...,0.731707,1,0.585366,1,0.524390,1,0.292683,0,0.353659,0
2,1610612738,Boston Celtics,2002-03,2003-04,2003,comparison,0,0.536585,0.439024,0.487805,...,0.548780,1,0.402439,0,0.292683,0,0.804878,1,0.756098,1
3,1610612739,Cleveland Cavaliers,2003-04,2004-05,2004,comparison,0,0.426829,0.512195,0.469512,...,0.609756,1,0.609756,1,0.548780,1,0.804878,1,0.743902,1
4,1610612739,Cleveland Cavaliers,2010-11,2011-12,2011,sustained_losing,1,0.231707,0.318182,0.274945,...,0.292683,0,0.402439,0,0.646341,1,0.695122,1,0.621951,1


In [5]:
summary = analysis.groupby("group").agg(
    examples=("team_id", "size"),
    unique_teams=("team_id", "nunique"),
    future_avg_win_pct=("future_avg_win_pct", "mean"),
    avg_playoff_appearances=("future_playoff_appearances", "mean"),
    probability_any_playoffs=("made_playoffs_within_5", "mean"),
)

display(summary.round(3))

,examples,unique_teams,future_avg_win_pct,avg_playoff_appearances,probability_any_playoffs
group,,,,,
comparison,41,27,0.476,2.415,0.854
sustained_losing,20,17,0.493,2.500,0.950
